# Chapter 2

In [3]:
import torch

print(f"PyTorch version {torch.__version__}")

if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")
    device = torch.device("mps")

else:
    print("Only CPU")

PyTorch version 2.10.0
Apple Silicon GPU


In [4]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

In [5]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_path)

In [6]:
prompt = "Tell me a story"
input_token_ids_list = tokenizer.encode(prompt)

In [7]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Tell me a story


In [8]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")


40451 --> Tell
752 -->  me
264 -->  a
3364 -->  story


In [9]:
gibberish = "Me kqwazak moe dubler dar a sturgi muz pho lafa wek gahmen"
input_token_ids_list = tokenizer.encode(gibberish)

for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

7823 --> Me
595 -->  k
69291 --> qw
1370 --> az
585 --> ak
4544 -->  mo
68 --> e
22113 -->  dub
1536 --> ler
15269 -->  dar
264 -->  a
357 -->  st
5556 --> urg
72 --> i
296 -->  m
5197 --> uz
1319 -->  ph
78 --> o
1187 -->  la
3632 --> fa
582 -->  we
74 --> k
342 -->  g
1466 --> ah
5676 --> men


In [10]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")
        device = torch.device("cuda")
        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"  # type: ignore
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True
    elif torch.backends.mps.is_available():
        print("Apple Silicon GPU")
        device = torch.device("mps")
    else:
        print("Only CPU")
        device = torch.device("cpu")
    return device

In [11]:
device = get_device()

Apple Silicon GPU


In [12]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [13]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)  # instantiate from random weights
model.load_state_dict(torch.load(model_path))  # load pretrained weights
model.to(device)  # transfer model in device memory

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [14]:
prompt = "Explain what an integral is."
input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list)  # convert list to tensor
input_tensor_fmt = input_tensor.unsqueeze(0)  # add a dim
input_tensor_fmt = input_tensor_fmt.to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt)  # generate output
output_tensor_fmt = output_tensor.squeeze(0)  # remove the dim
print(f"Formatted Output tensor shape: {output_tensor_fmt.shape}")

Number of input tokens: 7
Formatted Output tensor shape: torch.Size([7, 151936])


### A note on squeezing and unsqueezing Tensors
The pytorch methods `.squeeeze()` and `.unsqueeze()` are useful for resizing tensors to be compatible with other methods or objects that expect a specific dimension, such as the model above which expects two dimensions for rows and columns. The methods always add a dimension of size 1 from the position specified, so in our usage that is 0 when squeezed it adds the dimension and unsqueezing removes this dimension.

In [15]:
print(output_tensor_fmt)

tensor([[ 7.4062, 11.4375,  9.2500,  ...,  3.7500,  3.7500,  3.7500],
        [ 9.4375, 10.6875,  7.2812,  ...,  3.2344,  3.2344,  3.2344],
        [ 8.4375, 11.0000,  5.0312,  ...,  2.5469,  2.5469,  2.5469],
        ...,
        [ 8.6250, 12.7500, 10.6875,  ..., -0.6016, -0.6016, -0.6016],
        [15.5000, 12.9375, 10.6250,  ...,  0.3340,  0.3340,  0.3340],
        [ 9.5000,  5.5000,  7.8438,  ..., -1.4453, -1.4453, -1.4453]],
       device='mps:0', dtype=torch.bfloat16)


In order to get the output we take the last row from this formatted output tensor and find the max value, since the row represents a distribution of potential outputs (a score for every token in the vocabulary) - the max score is the most likely output.

In [16]:
last_row = output_tensor_fmt[-1]
print(last_row)

tensor([ 9.5000,  5.5000,  7.8438,  ..., -1.4453, -1.4453, -1.4453],
       device='mps:0', dtype=torch.bfloat16)


In [17]:
max_token_id = torch.argmax(last_row, dim=-1, keepdim=True)
print(max_token_id)

tensor([2585], device='mps:0')


In [18]:
output = tokenizer.decode([max_token_id])
print(output)

 How


#### Max vs. Argmax
Here we used `torch.argmax` over `torch.max` because the former returns the index of the larges value in a tensor while the latter returns its value.

In [19]:
def text_generation(input_token_ids_list, end_token_ids_list):
    while end_token_ids_list not in input_token_ids_list:
        input_tensor = torch.tensor(input_token_ids_list)  # convert list to tensor
        input_tensor_fmt = input_tensor.unsqueeze(0)  # add a dim
        input_tensor_fmt = input_tensor_fmt.to(device)

        with torch.inference_mode():
            output_tensor = model(input_tensor_fmt)  # generate output
        output_tensor_fmt = output_tensor.squeeze(0)  # remove the dim

        last_row = output_tensor_fmt[-1]

        next_token_id = torch.argmax(last_row, dim=-1, keepdim=True)
        input_token_ids_list.append(next_token_id)

        return text_generation(input_token_ids_list, end_token_ids_list)
    return input_token_ids_list

In [20]:
"""
end = "<|im_end|>"
end_token_ids_list = tokenizer.encode(end)
prompt = "Explain an integral in two sentences."
input_token_ids_list = tokenizer.encode(prompt)

 output_token_ids_list = text_generation(input_token_ids_list, end_token_ids_list)

 for i in output_token_ids_list:
    print(tokenizer.decode([i]))
"""

'\nend = "<|im_end|>"\nend_token_ids_list = tokenizer.encode(end)\nprompt = "Explain an integral in two sentences."\ninput_token_ids_list = tokenizer.encode(prompt)\n\n output_token_ids_list = text_generation(input_token_ids_list, end_token_ids_list)\n\n for i in output_token_ids_list:\n    print(tokenizer.decode([i]))\n'

Above is my paltry attempt at writing my own function before reading the book implementation. It does what it needs to (autoregressively generate next tokens) but it runs out of memory before it can finish... 

In [21]:
@torch.inference_mode()
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):
    model.eval()

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1]
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if (eos_token_id is not None) and torch.all(next_token == eos_token_id):
            break

        yield next_token  # return the next token immediately for streaming

        token_ids = torch.cat([token_ids, next_token], dim=1)

In [22]:
prompt = "Explain cats in a single sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt), device=device
).unsqueeze(0)

max_new_tokens = 100

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)

 Cats are small, independent, and often misunderstood domesticated animals known for their independent nature and affection for humans.

Annnnd I still ran out of memory lol

> RuntimeError: MPS backend out of memory (MPS allocated: 30.17 GiB, other allocations: 9.39 MiB, max allowed: 30.19 GiB). Tried to allocate 13.33 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).


Turns out I had to restart the kernel the old traceback and tensors from my runaway loop was holding it up.

In [23]:
import warnings


def generate_stats(output_token_ids, tokenizer, start_time, end_time):
    total_time = end_time - start_time
    print(f"\nTime: {total_time:.2f} sec")
    print(f"{int(output_token_ids.numel() / total_time)} avg tokens/sec")

    for name, backend in (
        ("CUDA", getattr(torch, "cuda", None)),
        ("XPU", getattr(torch, "xpu", None)),
    ):
        if backend is not None and backend.is_available():
            device_type = output_token_ids.device.type  # 1
            if device_type != name.lower():
                warnings.warn(
                    f"{name} is available but tensors are on "
                    f"{device_type}. Memory stats may be 0."
                )
            if hasattr(backend, "synchronize"):  # 2
                backend.synchronize()
            max_mem_bytes = backend.max_memory_allocated()
            max_mem_gb = max_mem_bytes / (1024**3)
            print(f"Max {name} memory allocated: {max_mem_gb:.2f} GB")
            backend.reset_peak_memory_stats()

In [ ]:
import time

start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id,
):
    token_id = token.squeeze(0).tolist()
    print(tokenizer.decode(token_id), end="", flush=True)
    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)

end_time = time.time()
output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(
    output_token_ids_tensor,
    tokenizer,
    start_time,
    end_time,
)

TypeError: 'int' object is not an instance of 'Sequence'